# One reproducible D2 QSAR workflow
**Human dopamine D2 binding assay CHEMBL3855278 · 80 original experimental pKi observations · saved 57/23 split · Ridge α=1 versus training mean**

This is a scientifically adapted integration of Datamol inspection and Patrick Walters’ dataset-splitting ideas—not a literal replay of his notebook. All scientific operations are visible below. **Flow: verify source records → preserve identities and pKi → calculate five features → load the saved split → fit on training rows → reproduce reference-test results → inspect chemistry → export.** We retain the original cohort, split and models; this is reproduction, not a search for a better score.

The 23 reference-test outcomes have already been inspected. Reproducing them is an audit of the original pilot, not fresh external validation. All results remain exploratory, from one assay and document. No claim of a useful prospective model is implied.

## Setup
In Colab, choose **Runtime → Change runtime type → Runtime version 2026.07 (Python 3.12)** and CPU, then run from the first cell. Outside Colab, use a **fresh Python 3.12 kernel**. Setup checks package versions, installs the pinned stack from official PyPI if needed, and downloads hash-verified public data. The long version-lock cell is collapsed for readability; expand it to inspect the exact pins.

Colab may preload NumPy. If packages need installation, setup installs them and deliberately stops: use **Runtime → Restart session**, then **Run all** from the top. Setup verifies both installed and already-loaded scientific versions before continuing; do not skip this stop or continue with mixed in-memory versions. Installation may take several minutes; the scientific calculations take seconds. No downloaded scientific Python module is executed.

Each run writes a new `coherent_d2_qsar_…` folder in the current working directory. The last cell lists the files and creates a downloadable results ZIP. In Colab, use the left **Files** panel to find and download that ZIP. A verified local source folder can be used offline once packages are installed.

In [ ]:
from pathlib import Path, PurePosixPath
import sys, os, json, hashlib, urllib.request, importlib.metadata as metadata, subprocess, tempfile
assert sys.version_info[:2] == (3, 12), "Use a fresh Python 3.12 runtime for this pinned workflow"
SOURCE_COMMIT = "6d07e2ab78b54cc6aeb44523101a4672aa159705"
SOURCE_BASE = f"https://raw.githubusercontent.com/DhruvMenon2003/drug-hansch-space/{SOURCE_COMMIT}/notebooks/qsar/chembl3855278/"
MANIFEST_SHA256 = "cfb8b096f3c00c2be092993bbbfd90f1a15d8cbdcda6fb3f910dc508360f2cf9"
SOURCE = Path(os.environ.get("QSAR_SOURCE", "coherent_qsar_source")).absolute()
OUTPUT = Path(tempfile.mkdtemp(prefix="coherent_d2_qsar_", dir=Path.cwd()))
for key, folder in [("MPLCONFIGDIR", "mpl"), ("PYSTOW_HOME", "pystow"), ("XDG_CACHE_HOME", "cache")]:
    path = OUTPUT / folder
    path.mkdir()
    os.environ[key] = str(path)
def sha_bytes(data): return hashlib.sha256(data).hexdigest()
def sha_file(path): return sha_bytes(Path(path).read_bytes())

In [ ]:
PINNED_REQUIREMENTS = 'aiobotocore==3.9.2\naiohappyeyeballs==2.7.1\naiohttp==3.14.5\naioitertools==0.13.0\naiosignal==1.4.0\nanyio==4.15.1\nargon2-cffi==25.1.0\nargon2-cffi-bindings==26.1.0\narrow==1.4.0\nasttokens==3.0.2\nasync-lru==2.4.0\nattrs==26.1.0\nbabel==2.18.0\nbackports.zstd==1.8.0\nbeautifulsoup4==4.15.0\nbleach==6.4.0\nbotocore==1.43.106\ncertifi==2026.10.19\ncffi==2.1.1\ncharset-normalizer==3.5.2\ncloudpickle==3.1.2\ncomm==0.2.3\ncontourpy==1.4.0\ncryptography==50.0.2\ncycler==0.12.1\ndatamol==0.13.0\ndebugpy==1.8.22\ndecorator==5.3.1\ndefusedxml==0.7.1\net_xmlfile==2.0.0\nexecuting==2.3.0\nfastjsonschema==2.22.2\nfonttools==4.67.0\nformulaic==1.2.2\nfqdn==1.6.0\nfrozenlist==1.8.0\nfsspec==2026.9.0\ngcsfs==2026.10.1\ngoogle-api-core==2.43.0\ngoogle-auth==2.62.0\ngoogle-auth-oauthlib==1.5.0\ngoogle-cloud-core==2.8.0\ngoogle-cloud-storage==3.17.1\ngoogle-cloud-storage-control==1.17.0\ngoogle-crc32c==1.9.0\ngoogle-resumable-media==2.11.0\ngoogleapis-common-protos==1.76.0\ngrpc-google-iam-v1==0.14.6\ngrpcio==1.84.1\ngrpcio-status==1.84.1\nh11==0.16.0\nhttpcore==1.0.9\nhttpx==0.28.1\nidna==3.20\ninterface_meta==2.0.1\nipykernel==7.4.0\nipython==9.17.1\nipython_pygments_lexers==1.1.1\nipywidgets==8.1.9\nisoduration==20.11.0\njedi==0.20.1\nJinja2==3.1.6\njmespath==1.1.0\njoblib==1.6.0\njson5==0.17.3\njsonpointer==3.4\njsonschema==4.26.0\njsonschema-specifications==2025.9.1\njupyter-events==0.12.1\njupyter-lsp==2.3.1\njupyter_builder==1.2.3\njupyter_client==8.10.0\njupyter_core==5.9.1\njupyter_server==2.21.1\njupyter_server_terminals==0.5.4\njupyterlab==4.6.4\njupyterlab_pygments==0.3.0\njupyterlab_server==2.28.1\njupyterlab_widgets==3.0.17\nkiwisolver==1.5.1\nlark==1.3.1\nloguru==0.7.3\nMarkupSafe==3.0.4\nmatplotlib==3.11.2\nmatplotlib-inline==0.2.2\nmistune==3.4.0\nmultidict==6.9.1\nnarwhals==2.27.1\nnbclient==0.11.0\nnbconvert==7.17.2\nnbformat==5.11.1\nnest-asyncio2==1.7.4\nnetworkx==3.7\nnglview==4.0.1\nnotebook==7.6.3\nnotebook_shim==0.2.4\nnumpy==2.5.4\noauthlib==4.0.0\nopenpyxl==3.1.5\nopentelemetry-api==1.45.1\npackaging==26.3\npandas==3.1.0\npandocfilters==1.5.1\nparso==0.8.7\npatsy==1.0.3\npexpect==4.9.0\npillow==13.0.0\nplatformdirs==4.13.5\nprometheus_client==0.26.0\nprompt_toolkit==3.0.53\npropcache==0.5.4\nproto-plus==1.29.0\nprotobuf==7.36.2\npsutil==7.2.2\nptyprocess==0.7.0\npure_eval==0.2.4\npy3Dmol==2.5.5\npyarrow==26.0.0\npyasn1==0.6.4\npyasn1_modules==0.4.2\npycparser==3.12\nPygments==2.21.0\npyparsing==3.3.3\npystow==0.10.3\npython-dateutil==2.9.0.post0\npython-json-logger==4.2.0\nPyYAML==6.0.3\npyzmq==27.2.0\nrdkit==2026.9.1\nreferencing==0.37.0\nrequests==2.34.2\nrequests-oauthlib==2.0.0\nrfc3339-validator==0.1.4\nrfc3986-validator==0.1.1\nrfc3987-syntax==1.1.0\nrpds-py==2026.9.1\ns3fs==2026.9.0\nscikit-learn==1.9.1\nscipy==1.18.1\nseaborn==0.13.2\nselfies==2.2.0\nSend2Trash==2.1.0\nsix==1.17.0\nsoupsieve==3.0.4\nstack-data==0.6.3\nstatsmodels==0.15.0\nterminado==0.18.1\nthreadpoolctl==3.7.0\ntinycss2==1.5.1\ntornado==6.6\ntqdm==4.70.1\ntraitlets==5.16.1\ntyping_extensions==4.16.0\ntzdata==2026.5\nuri-template==1.3.0\nurllib3==2.8.0\nuseful_rdkit_utils==2.0.1\nwcwidth==0.9.3\nwebcolors==25.10.0\nwebencodings==0.6.1\nwebsocket-client==1.9.2\nwidgetsnbextension==4.0.16\nwrapt==2.5.1\nxlsxwriter==3.3.0\nyarl==1.25.1\n'
requirements = dict(line.split("==", 1) for line in PINNED_REQUIREMENTS.splitlines() if line.strip())
def installed_version(name):
    try: return metadata.version(name)
    except metadata.PackageNotFoundError: return None
mismatched = {name: (installed_version(name), version) for name, version in requirements.items()
              if installed_version(name) != version}
# Install on disk if needed, then require a supported kernel restart before any science.
if mismatched:
    requirement_path = OUTPUT / "requirements-science.lock.txt"
    requirement_path.write_text(PINNED_REQUIREMENTS)
    pip_env = {key:value for key,value in os.environ.items() if not key.startswith("PIP_")}
    pip_env["PIP_CONFIG_FILE"] = os.devnull
    subprocess.run([sys.executable, "-m", "pip", "install", "--index-url", "https://pypi.org/simple", "-r", str(requirement_path)], env=pip_env, check=True)
    assert all(installed_version(name) == version for name, version in requirements.items())
    raise RuntimeError("Pinned package installation completed. REQUIRED: Runtime > Restart session, then Run all from the top. No data or modeling cells have run; do not skip this restart.")
assert all(installed_version(name) == version for name, version in requirements.items())
# A restarted Colab session can legitimately preload matching NumPy. Reject stale modules.
from packaging.version import Version
module_distributions = {"numpy":"numpy", "pandas":"pandas", "rdkit":"rdkit",
    "sklearn":"scikit-learn", "datamol":"datamol", "useful_rdkit_utils":"useful_rdkit_utils",
    "scipy":"scipy", "matplotlib":"matplotlib", "seaborn":"seaborn", "statsmodels":"statsmodels", "joblib":"joblib"}
stale_loaded = {}
for module_name, distribution in module_distributions.items():
    if module_name in sys.modules:
        loaded_version = getattr(sys.modules[module_name], "__version__", None)
        try:
            matches = loaded_version is not None and Version(str(loaded_version)) == Version(requirements[distribution])
        except (TypeError, ValueError):
            matches = False
        if not matches: stale_loaded[module_name] = {"loaded":loaded_version, "required":requirements[distribution]}
if stale_loaded:
    raise RuntimeError(f"Stale in-memory scientific modules: {stale_loaded}. Runtime > Restart session, then Run all; no scientific work can continue in this session.")
subprocess.run([sys.executable, "-m", "pip", "check"], check=True)
print("Pinned scientific environment verified; Python", sys.version.split()[0])

In [ ]:
DATA_FILES = ['DATA_LICENSE.md', 'frozen/eligible_compounds.csv', 'frozen/decision_ledger.csv', 'frozen/exclusion_ledger.csv', 'frozen/duplicate_ledger.csv', 'frozen/censoring_and_missing_ledger.csv', 'frozen/all_activity_rows.csv', 'frozen/cohort_rules.json', 'frozen/cohort_summary.json', 'frozen/source_manifest.json', 'frozen/raw/assay.json', 'frozen/raw/target.json', 'frozen/raw/document.json', 'frozen/raw/activities_page_000.json', 'predeclared_recipe.json', 'prefit_split_manifest.csv', 'prefit_split_info.json', 'results/model_dataset.csv', 'results/predictions.csv', 'results/report.json', 'results/model_parameters.json']
def safe_relative(name):
    p = PurePosixPath(name)
    if p.is_absolute() or ".." in p.parts or "\\" in name:
        raise ValueError(f"Unsafe relative path: {name}")
    return p

def fetch_bytes(name):
    safe_relative(name)
    with urllib.request.urlopen(SOURCE_BASE + name, timeout=90) as response:
        return response.read()

if SOURCE.resolve() != SOURCE:
    raise ValueError("Symlinked source folder rejected")
SOURCE.mkdir(parents=True, exist_ok=True)
manifest_path = SOURCE / "FINAL_MANIFEST.json"
manifest_raw = manifest_path.read_bytes() if manifest_path.exists() else fetch_bytes("FINAL_MANIFEST.json")
assert sha_bytes(manifest_raw) == MANIFEST_SHA256, "Immutable manifest mismatch"
source_manifest = json.loads(manifest_raw)
for name in DATA_FILES:
    path = SOURCE / safe_relative(name)
    assert path.resolve() == path, "Symlinked input rejected"
    data = path.read_bytes() if path.exists() else fetch_bytes(name)
    assert sha_bytes(data) == source_manifest[name], f"Input changed: {name}"
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(data)
if not manifest_path.exists(): manifest_path.write_bytes(manifest_raw)
def verify_inputs():
    assert sha_file(manifest_path) == MANIFEST_SHA256
    for name in DATA_FILES:
        assert sha_file(SOURCE / name) == source_manifest[name], name
verify_inputs()
print(f"Verified {len(DATA_FILES)} immutable data/provenance files. No external Python module executed.")

## 1. One observation, one identity, one response
The frozen curation began with **107 assay records** and retained **80 compounds**. It excluded all multiply recorded full-parent identities rather than choosing or averaging their labels; overlapping exclusion reasons are not additive. Parent cleanup/fragment selection/uncharging happened once in that recorded curation, not here. Source and parent structures, activity identifiers, rules and the complete exclusion ledger remain available.

Our response is **original experimental pKi**, not rounded ChEMBL pChEMBL or a reverse-converted Ki value. `Ki(nM)=10**(9-pKi)` is a derived consistency check, not a second experiment. No row is filtered by molecular properties or clustering. We audit the frozen curation result against its raw snapshot; we do not silently re-curate it under a newer toolkit.

In [ ]:
import numpy as np
import pandas as pd
import datamol as dm
import useful_rdkit_utils as uru
from rdkit import Chem, DataStructs
from rdkit.Chem import Descriptors, rdFingerprintGenerator
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
from IPython.display import display

cohort = pd.read_csv(SOURCE / "frozen/eligible_compounds.csv")
ledger = pd.read_csv(SOURCE / "frozen/decision_ledger.csv")
excluded = pd.read_csv(SOURCE / "frozen/exclusion_ledger.csv")
summary = json.loads((SOURCE / "frozen/cohort_summary.json").read_text())
assert len(cohort) == 80 and len(ledger) == 107 and len(excluded) == 27
assert cohort.molecule_chembl_id.is_unique and cohort.parent_inchikey.is_unique
expected = {"assay_chembl_id":"CHEMBL3855278", "target_chembl_id":"CHEMBL217",
            "document_chembl_id":"CHEMBL3853329", "original_type":"pKi", "original_relation":"=",
            "standard_type":"Ki", "standard_units":"nM", "standard_relation":"=",
            "label_origin":"source_reported_pKi", "decision":"include"}
for column, value in expected.items(): assert cohort[column].eq(value).all(), column
assert np.array_equal(cohort.pKi.to_numpy(), cohort.original_value.to_numpy())
assert np.isfinite(cohort.pKi).all() and cohort.source_records_for_full_identity.eq(1).all()
included_ids = set(ledger.loc[ledger.audit_decision.eq("include"), "activity_id"].astype(str))
assert included_ids == set(cohort.activity_id.astype(str))
assert not included_ids.intersection(set(excluded.activity_id.astype(str)))
assert set(ledger.activity_id.astype(str)) == included_ids.union(set(excluded.activity_id.astype(str)))
raw_records = json.loads((SOURCE / "frozen/raw/activities_page_000.json").read_text())["activities"]
raw_by_id = {str(record["activity_id"]):record for record in raw_records}
assert len(raw_records) == len(raw_by_id) == 107
assert set(raw_by_id) == set(ledger.activity_id.astype(str))
for row in cohort.itertuples():
    original = raw_by_id[str(row.activity_id)]
    assert original["molecule_chembl_id"] == row.molecule_chembl_id
    assert original["type"] == "pKi" and original["relation"] == "="
    assert original["standard_type"] == "Ki" and original["standard_relation"] == "="
    assert original["standard_units"] == "nM"
    assert float(original["value"]) == row.pKi
    assert original["canonical_smiles"] == row.source_canonical_smiles
print("107 source records → 80 retained compounds; 27 excluded rows. Exclusion counts overlap:")
display(pd.Series(summary["exclusion_reason_counts"], name="count").to_frame())
display(excluded[["activity_id", "molecule_chembl_id", "audit_exclusion_reasons"]].head())

molecules = dm.from_df(cohort[["molecule_chembl_id", "parent_smiles"]].copy(),
                       smiles_column="parent_smiles", conserve_smiles=True, sanitize=True)
assert all(mol is not None for mol in molecules)
assert [Chem.MolToInchiKey(mol) for mol in molecules] == cohort.parent_inchikey.tolist()
identity_before = [Chem.MolToSmiles(mol, isomericSmiles=True) for mol in molecules]
observations = cohort.copy()
observations["derived_Ki_nM"] = dm.molar.log_to_molar(cohort.pKi.to_numpy(), "nM")
assert np.allclose(observations.derived_Ki_nM, 10**(9-cohort.pKi), rtol=1e-12)
assert (abs(observations.derived_Ki_nM - cohort.standard_Ki_nM) <= cohort.conversion_tolerance_nM).all()
observations["connectivity_key"] = [Chem.MolToSmiles(mol, isomericSmiles=False) for mol in molecules]
observations["scaffold"] = [MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False) for mol in molecules]
assert observations.connectivity_key.nunique() == 60 and observations.scaffold.nunique() == 32

## 2. Five interpretable features, checked by a second calculation path
The model receives only **MolWt, MolLogP, TPSA, NumHDonors, NumHAcceptors**, in that order. useful_rdkit_utils supplies the production calculation; Datamol batches the same RDKit functions through a separate calculation path. This checks values, feature ordering and plumbing—not independent scientific validation of RDKit descriptor definitions. No descriptor selection, imputation, scaling or fitting happens yet. Rule-of-Five flags are inspection only.

In [ ]:
FEATURES = ["MolWt", "MolLogP", "TPSA", "NumHDonors", "NumHAcceptors"]
calculator = uru.RDKitDescriptors(desc_names=FEATURES, hide_progress=True)
X = np.asarray([calculator.calc_mol(mol) for mol in molecules], dtype=float)
assert X.shape == (80, 5) and np.isfinite(X).all()
independent_features = dm.descriptors.batch_compute_many_descriptors(
    molecules, properties_fn={name: getattr(Descriptors, name) for name in FEATURES},
    add_properties=False, n_jobs=1, progress=False)
assert np.allclose(X, independent_features[FEATURES].to_numpy(), rtol=0, atol=1e-12)
for index, name in enumerate(FEATURES): observations[name] = X[:, index]
reference_features = pd.read_csv(SOURCE / "results/model_dataset.csv").set_index("molecule_chembl_id").loc[cohort.molecule_chembl_id]
assert np.allclose(X, reference_features[FEATURES].to_numpy(), rtol=0, atol=1e-12)
observations["ro5_criteria_met"] = ((observations.MolWt <= 500).astype(int) +
    (observations.MolLogP <= 5).astype(int) + (observations.NumHDonors <= 5).astype(int) +
    (observations.NumHAcceptors <= 10).astype(int))
assert len(observations) == 80
feature_table = observations[["molecule_chembl_id"] + FEATURES].copy()
display(feature_table.head())

## 3. One saved split, shared by every calculation
The original split grouped parent, connectivity and nonchiral Murcko scaffold identities. Groups were ordered by a fixed SHA256 seed without looking at pKi. Allocating whole groups overshot the 16-compound target, giving **57 train / 23 reference test**. We use the exact saved identities—not a new shuffle. The original recipe is displayed and archived; overlap assertions independently validate its grouping.

Every prediction, nearest-neighbor calculation and structural diagnostic below uses these same masks. The test labels have already been seen historically; this run reproduces that experiment, not a new untouched test.

In [ ]:
recipe = json.loads((SOURCE / "predeclared_recipe.json").read_text())
split_info = json.loads((SOURCE / "prefit_split_info.json").read_text())
saved_split = pd.read_csv(SOURCE / "prefit_split_manifest.csv")
assert saved_split.molecule_chembl_id.is_unique
assert set(saved_split.molecule_chembl_id) == set(cohort.molecule_chembl_id)
ordered_split = saved_split.set_index("molecule_chembl_id").loc[cohort.molecule_chembl_id]
for column in ["connectivity_key", "scaffold"]:
    assert ordered_split[column].tolist() == observations[column].tolist(), column
observations["split"] = ordered_split.split.to_numpy()
assert set(observations.split) == {"train", "test"}
train = observations.split.eq("train").to_numpy()
test = observations.split.eq("test").to_numpy()
assert train.sum() == 57 and test.sum() == 23 and np.all(train ^ test)
for key in ["parent_molecule_chembl_id", "connectivity_key", "scaffold"]:
    assert not set(observations.loc[train, key]) & set(observations.loc[test, key]), key
assert np.array_equal(observations.split, reference_features.split.to_numpy())
SPLIT_SHA256 = sha_file(SOURCE / "prefit_split_manifest.csv")
saved_split.to_csv(OUTPUT / "split_manifest.csv", index=False)
print(recipe["split"])
print("Verified exact 57/23 membership; no parent/connectivity/scaffold overlap.")

## 4. Fit only on training rows; reproduce both original models
The scaler and Ridge α=1 are one pipeline. Both it and the training-mean baseline learn only from the 57 training observations. No hyperparameter search, alternative split, model selection or all-data refit occurs. Training predictions are labeled in-sample, never mixed with test metrics.

In [ ]:
y = observations.pKi.to_numpy(dtype=float)
ridge = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
ridge.fit(X[train], y[train])
baseline = DummyRegressor(strategy="mean").fit(X[train], y[train])
assert np.allclose(ridge[0].mean_, X[train].mean(axis=0), rtol=0, atol=1e-12)
assert ridge[0].n_samples_seen_ == 57
predictions = observations.copy()
predictions["observed_pKi"] = y
predictions["observation_kind"] = "source_reported_experimental_pKi"
predictions["evaluation_role"] = np.where(train, "training_in_sample", "previously_observed_reference_test")
predictions["predicted_pKi_ridge"] = ridge.predict(X)
predictions["predicted_pKi_train_mean"] = baseline.predict(X)
predictions["residual_observed_minus_predicted"] = y - predictions.predicted_pKi_ridge
assert np.isfinite(predictions[["predicted_pKi_ridge", "predicted_pKi_train_mean"]]).all().all()
metrics = {}
for model, column in [("training_mean", "predicted_pKi_train_mean"), ("ridge_alpha_1", "predicted_pKi_ridge")]:
    pred = predictions.loc[test, column].to_numpy()
    metrics[model] = {"test_n":int(test.sum()), "MAE":float(mean_absolute_error(y[test], pred)),
        "RMSE":float(np.sqrt(mean_squared_error(y[test], pred))), "R2":float(r2_score(y[test], pred))}
reference_predictions = pd.read_csv(SOURCE / "results/predictions.csv").set_index("molecule_chembl_id").loc[cohort.molecule_chembl_id]
for column in ["predicted_pKi_ridge", "predicted_pKi_train_mean"]:
    assert np.allclose(predictions[column], reference_predictions[column], rtol=0, atol=1e-10), column
reference_report = json.loads((SOURCE / "results/report.json").read_text())
for model, values in metrics.items():
    for metric, value in values.items():
        assert abs(value - reference_report["holdout_metrics"][model][metric]) < 1e-10, (model, metric)
parameters = {"features":FEATURES, "scaler_mean":ridge[0].mean_.tolist(),
    "scaler_scale":ridge[0].scale_.tolist(), "ridge_coef":ridge[1].coef_.tolist(),
    "ridge_intercept":float(ridge[1].intercept_), "baseline_mean":float(baseline.constant_.item())}
replayed = ((X-np.array(parameters["scaler_mean"]))/np.array(parameters["scaler_scale"])) @ np.array(parameters["ridge_coef"]) + parameters["ridge_intercept"]
assert np.allclose(replayed, predictions.predicted_pKi_ridge, rtol=0, atol=1e-12)
display(pd.DataFrame(metrics).T)

## 5. Shared-split structural diagnostics—not a resplitting experiment
We keep representation definitions explicit. The original pilot's applicability diagnostics use Morgan **radius 2 / 2048 bits / nonchiral** fingerprints; descriptor ranges are learned from train only. These flags are descriptive, not calibrated confidence.

The Walters default Butina diagnostic uses **radius 2 / 1024 bits / distance cutoff 0.65**. On this cohort it produces two groups (79 and 1). Therefore five nonempty grouped test folds cannot exist. We report **not evaluable** instead of changing the cutoff, reducing folds or silently replacing the algorithm. Its cluster table is crossed against the *saved* split; it never changes that split.

Nonchiral representations cannot resolve every stereochemical identity. Shared representations with different pKi values are a modeling/provenance limitation, not proof that stereochemistry caused the activity difference. No observations are averaged, dropped or automatically made chiral.

In [ ]:
pilot_fpgen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048, includeChirality=False)
pilot_fps = [pilot_fpgen.GetFingerprint(mol) for mol in molecules]
train_indices = np.flatnonzero(train)
predictions["outside_training_descriptor_range"] = ((X < X[train].min(axis=0)) | (X > X[train].max(axis=0))).any(axis=1)
predictions["max_train_morgan_similarity"] = [
    max(DataStructs.TanimotoSimilarity(pilot_fps[i], pilot_fps[j]) for j in train_indices if j != i)
    for i in range(len(molecules))]
assert np.array_equal(predictions.outside_training_descriptor_range, reference_predictions.outside_training_descriptor_range)
assert np.allclose(predictions.max_train_morgan_similarity, reference_predictions.max_train_morgan_similarity, rtol=0, atol=1e-12)

# Exact current public helper default: radius 2, 1024 bits, distance cutoff 0.65.
butina_groups = np.asarray(uru.get_butina_clusters(cohort.parent_smiles))
butina_sizes = pd.Series(butina_groups).value_counts().sort_values(ascending=False).tolist()
assert butina_sizes == [79, 1]
cluster_table = pd.DataFrame({"molecule_chembl_id":cohort.molecule_chembl_id,
    "default_butina_group":butina_groups, "saved_split":observations.split})
butina_status = {"status":"not_evaluable_as_five_nonempty_grouped_folds", "group_sizes":butina_sizes,
    "distance_cutoff":0.65, "radius":2, "fpSize":1024, "includeChirality":False,
    "folds_requested_by_upstream":5, "folds_run":0, "split_changed":False}
display(pd.crosstab(cluster_table.default_butina_group, cluster_table.saved_split))
print(butina_status)

upstream_fpgen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024, includeChirality=False)
bit_keys = [upstream_fpgen.GetFingerprint(mol).ToBitString() for mol in molecules]
count_keys = [tuple(upstream_fpgen.GetCountFingerprintAsNumPy(mol).tolist()) for mol in molecules]
assert len(set(bit_keys)) == len(set(count_keys)) == 60
representation = observations[["molecule_chembl_id", "parent_inchikey", "connectivity_key", "pKi", "split"]].copy()
representation["nonchiral_bit_group"] = pd.factorize(pd.Series(bit_keys))[0]
representation["nonchiral_count_group"] = pd.factorize(pd.Series(count_keys, dtype=object))[0]
assert representation.groupby("nonchiral_bit_group").connectivity_key.nunique().max() == 1
assert representation.groupby("connectivity_key").nonchiral_bit_group.nunique().max() == 1
assert representation.groupby("nonchiral_count_group").connectivity_key.nunique().max() == 1
assert representation.groupby("nonchiral_bit_group").split.nunique().max() == 1
representation_summary = representation.groupby("nonchiral_bit_group").agg(n=("pKi","size"),
    pKi_min=("pKi","min"), pKi_max=("pKi","max"))
representation_summary["observed_pKi_range"] = representation_summary.pKi_max - representation_summary.pKi_min
print("80 full identities / 60 nonchiral representations; no representation crosses the saved split.")
display(representation_summary[representation_summary.n > 1])

## 6. Read the results honestly
The original Ridge errors are only slightly smaller than the training-mean baseline; both test R² values are negative. This is not evidence of a successful prospective predictor. The plots below use only the same 23 reference-test identities. No Tukey tests, independence claims from repeated folds or newly selected “best split” are made.

In [ ]:
heldout = predictions.loc[test].copy()
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].scatter(heldout.observed_pKi, heldout.predicted_pKi_ridge)
lo = min(heldout.observed_pKi.min(), heldout.predicted_pKi_ridge.min())
hi = max(heldout.observed_pKi.max(), heldout.predicted_pKi_ridge.max())
axes[0].plot([lo,hi], [lo,hi], "k--")
axes[0].set(xlabel="Observed experimental pKi", ylabel="Predicted pKi", title="23 reference-test compounds")
axes[1].scatter(heldout.predicted_pKi_ridge, heldout.residual_observed_minus_predicted)
axes[1].axhline(0, color="k", linestyle="--")
axes[1].set(xlabel="Predicted pKi", ylabel="Observed − predicted", title="Same saved test split")
axes[2].scatter(heldout.max_train_morgan_similarity, abs(heldout.residual_observed_minus_predicted))
axes[2].set(xlabel="Maximum training similarity (r2/2048 bits)", ylabel="Absolute pKi error", title="Descriptive—not calibrated confidence")
fig.tight_layout()
fig.savefig(OUTPUT / "shared_split_diagnostics.png", dpi=150)
plt.show()
display(heldout[["molecule_chembl_id", "observed_pKi", "predicted_pKi_ridge",
    "predicted_pKi_train_mean", "outside_training_descriptor_range", "max_train_morgan_similarity"]])

## 7. Export a replayable audit trail
Tables keep observations, predictions, identities and roles separate. JSON coefficients replay Ridge without deserializing a pickle. Input hashes, package versions, the exact split, protocol and an output manifest accompany the results. The receipt describes this kernel truthfully; it does not claim a Google Colab run merely because the notebook is Colab-compatible.

In [ ]:
assert [Chem.MolToSmiles(mol, isomericSmiles=True) for mol in molecules] == identity_before
assert len(predictions) == 80 and predictions.molecule_chembl_id.is_unique
assert np.array_equal(predictions.observed_pKi, cohort.original_value)
verify_inputs()
protocol = {"endpoint":"original source-reported experimental pKi", "cohort_n":80,
    "source_commit":SOURCE_COMMIT, "features":FEATURES, "split_sha256":SPLIT_SHA256,
    "train_n":57, "reference_test_n":23, "ridge_alpha":1.0, "baseline":"training_mean",
    "tuning":False, "resplitting":False, "cutoff_changed":False,
    "test_status":"previously observed reference test, not new external validation",
    "diagnostic_only_butina":butina_status}
observations.to_csv(OUTPUT / "observations_and_features.csv", index=False)
feature_table.to_csv(OUTPUT / "feature_matrix.csv", index=False)
predictions.to_csv(OUTPUT / "predictions.csv", index=False)
cluster_table.to_csv(OUTPUT / "default_butina_saved_split.csv", index=False)
representation.to_csv(OUTPUT / "representation_identity_audit.csv", index=False)
representation_summary.to_csv(OUTPUT / "representation_group_summary.csv")
for name, payload in [("protocol.json",protocol), ("metrics.json",metrics),
                      ("model_parameters.json",parameters), ("butina_status.json",butina_status)]:
    (OUTPUT / name).write_text(json.dumps(payload, indent=2)+"\n")
(OUTPUT / "source_data_manifest.json").write_text(json.dumps({"commit":SOURCE_COMMIT,
    "manifest_sha256":MANIFEST_SHA256, "files":{name:source_manifest[name] for name in DATA_FILES}}, indent=2)+"\n")
(OUTPUT / "requirements-science.lock.txt").write_text(PINNED_REQUIREMENTS)
versions = {name:installed_version(name) for name in requirements}
receipt = {"status":"passed", "python":sys.version, "versions":versions,
    "cohort_n":80, "train_n":57, "reference_test_n":23,
    "separate_calculation_path_features_match":True, "archived_predictions_metrics_reproduced":True,
    "saved_split_sha256":SPLIT_SHA256, "shared_split_diagnostics":True,
    "original_inputs_unchanged":True, "hidden_scientific_module_execution":False,
    "colab_runtime_detected": "google.colab" in sys.modules,
    "external_browser_verification_claimed":False}
(OUTPUT / "execution_receipt.json").write_text(json.dumps(receipt, indent=2)+"\n")
exports = {path.name:sha_file(path) for path in OUTPUT.iterdir() if path.is_file()}
(OUTPUT / "output_manifest.json").write_text(json.dumps(exports, indent=2)+"\n")
print("PASS: original predictions and metrics reproduced; immutable inputs verified.")
print("Outputs:", OUTPUT)
print("Exported files:")
for name in sorted(exports): print(" -", name)
# Downloadable result bundle; caches are excluded, and no file is deleted.
import zipfile
from IPython.display import FileLink
archive_path = Path.cwd() / (OUTPUT.name + ".zip")
with zipfile.ZipFile(archive_path, "x", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(OUTPUT.iterdir()):
        if path.is_file(): archive.write(path, arcname=path.name)
print("Results ZIP:", archive_path.name)
display(FileLink(archive_path.name))
print("Colab: open the left Files panel, refresh, then right-click this ZIP and choose Download.")
print("This is a results bundle; immutable input provenance and retrieval hashes are in source_data_manifest.json.")

## Sources, license and adaptation record
- Frozen assay package: [drug-hansch-space, commit 6d07e2ab](https://github.com/DhruvMenon2003/drug-hansch-space/tree/6d07e2ab78b54cc6aeb44523101a4672aa159705/notebooks/qsar/chembl3855278). ChEMBL source data and derived tables retain **CC BY-SA 3.0**; exact provenance and license are retrieved with the inputs. No full paper is redistributed.
- Patrick Walters: [dataset-splitting notebook](https://github.com/PatWalters/practical_cheminformatics_posts/blob/b4a74789beb1c43d18fe46f13bd5e77c1f2cce92/splitting/dataset_splitting.ipynb), pinned commit `b4a74789beb1c43d18fe46f13bd5e77c1f2cce92`; [Datamol tutorial repository](https://github.com/PatWalters/practical_cheminformatics_tutorials/tree/370dc66428819e06). The workflow uses their inspection/split-diagnostic ideas, not their original solubility outputs.
- **Adaptations approved for this notebook:** visible single-environment science, original pKi/cohort mapping, exact saved split rather than unrelated resampling, current useful_rdkit_utils descriptor API with independent RDKit checks, descriptive default Butina feasibility, explicit representation definitions, archived-result replay, shared-split diagnostics and source/output manifests. No historical `get_umap_clusters`, mixed-role `cross_validate`, LightGBM addition or Tukey inference is executed.
- This is an audit/reproduction of an exploratory single-assay pilot. Publication-grade predictive validity would require a separately specified prospective/external validation strategy; repeated use of these known labels does not create one.

### MIT license — upstream notebooks
Copyright (c) 2021 Patrick Walters (dataset-splitting notebook)

Copyright (c) 2022 Patrick Walters (Datamol tutorial)

Permission is hereby granted, free of charge, to any person obtaining a copy of this software and associated documentation files (the "Software"), to deal in the Software without restriction, including without limitation the rights to use, copy, modify, merge, publish, distribute, sublicense, and/or sell copies of the Software, and to permit persons to whom the Software is furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.